# Diplomski rad - Primena transformer arhitekture neuronskih mreza za detekciju DoS napada

## Beleznica 1: Preuzimanje podataka i eksploratorna analiza (EDA)

Prvi korak u razvoju modela zasnovanog na masinskom ucenju jeste obezbedjivanje odgovarajuceg skupa podataka i razumevanje njegove strukture. Ova beleznica obuhvata preuzimanje odabranog dataseta i sprovodjenje eksploratorne analize podataka (Exploratory Data Analysis, EDA) - postupka kojim se ispituju osnovne karakteristike podataka (broj instanci, broj i priroda atributa, raspodela klasa) pre bilo kakve dalje obrade.

### 1. Uvoz potrebnih biblioteka

In [ ]:
!pip install -q kagglehub

import kagglehub
import pandas as pd
import numpy as np
import os
from pathlib import Path

pd.set_option('display.max_columns', 100)

In [ ]:
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"

DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)
print("Results directory:", RESULTS_DIR)

### 2. Autentifikacija na Kaggle API

Za preuzimanje dataseta koristi se biblioteka kagglehub, koja zahteva odgovarajuce Kaggle kredencijale. Prilikom pokretanja u Google Colab okruzenju, kredencijali se ucitavaju iz Colab Secrets, dok se pri lokalnom pokretanju koriste kredencijali konfigurisani u lokalnom okruzenju. Na taj nacin API kljuc se ne cuva direktno u kodu niti u GitHub repozitorijumu.

In [ ]:
try:
    from google.colab import userdata

    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

    print("Kaggle credentials loaded from Colab Secrets.")

except ImportError:
    print("Local environment detected. Using locally configured Kaggle credentials.")

Kaggle kredencijali ucitani iz Colab Secrets.


### 3. Preuzimanje dataseta

U ovom radu koristi se javno dostupna, prethodno obradjena verzija dataseta CICIDS2017, publikovanog od strane Canadian Institute for Cybersecurity (CIC).

Iz ove verzije uklonjene su kolone poput IP adresa, portova i timestamp-a, koje bi mogle izazvati curenje podataka (data leakage) - model bi mogao da "nauci" te proizvoljne identifikatore umesto stvarnog ponasanja saobracaja.

In [ ]:
path = kagglehub.dataset_download("dhoogla/cicids2017")
print("Dataset preuzet u:", path)
print()
print("Fajlovi u datasetu:")
for f in sorted(os.listdir(path)):
    print(" -", f)

Using Colab cache for faster access to the 'cicids2017' dataset.
Dataset preuzet u: /kaggle/input/cicids2017

Fajlovi u datasetu:
 - Benign-Monday-no-metadata.parquet
 - Botnet-Friday-no-metadata.parquet
 - Bruteforce-Tuesday-no-metadata.parquet
 - DDoS-Friday-no-metadata.parquet
 - DoS-Wednesday-no-metadata.parquet
 - Infiltration-Thursday-no-metadata.parquet
 - Portscan-Friday-no-metadata.parquet
 - WebAttacks-Thursday-no-metadata.parquet


### 4. Ucitavanje relevantnog fajla

Snimanje podataka za originalni CICIDS2017 dataset sprovedeno je tokom pet radnih dana, pri cemu je svaki dan bio posvecen drugacijem tipu saobracaja.

Za potrebe ovog rada, ciji je predmet detekcija DoS napada, relevantan je iskljucivo fajl "DoS-Wednesday-no-metadata.parquet", koji obuhvata benigni saobracaj zabelezen tokom srede, kao i sve DoS napade izvedene tog dana tokom eksperimenta.


In [ ]:
dos_path = os.path.join(path, "DoS-Wednesday-no-metadata.parquet")
df = pd.read_parquet(dos_path)

print("Shape:", df.shape)

Shape: (584991, 78)


### 5. Prvi pregled podataka

Radi upoznavanja sa formatom, tipovima i rasponima vrednosti pojedinacnih atributa, prikazuje se pocetni segment ucitanog dataseta.

In [ ]:
df.head()

,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Fwd Packets Length Total,Bwd Packets Length Total,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,Bwd Packet Length Mean,Bwd Packet Length Std,Flow Bytes/s,Flow Packets/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Total,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Total,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Bwd PSH Flags,Fwd URG Flags,Bwd URG Flags,Fwd Header Length,Bwd Header Length,Fwd Packets/s,Bwd Packets/s,Packet Length Min,Packet Length Max,Packet Length Mean,Packet Length Std,Packet Length Variance,FIN Flag Count,SYN Flag Count,RST Flag Count,PSH Flag Count,ACK Flag Count,URG Flag Count,CWE Flag Count,ECE Flag Count,Down/Up Ratio,Avg Packet Size,Avg Fwd Segment Size,Avg Bwd Segment Size,Fwd Avg Bytes/Bulk,Fwd Avg Packets/Bulk,Fwd Avg Bulk Rate,Bwd Avg Bytes/Bulk,Bwd Avg Packets/Bulk,Bwd Avg Bulk Rate,Subflow Fwd Packets,Subflow Fwd Bytes,Subflow Bwd Packets,Subflow Bwd Bytes,Init Fwd Win Bytes,Init Bwd Win Bytes,Fwd Act Data Packets,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,6,38308,1,1,6,6,6,6,6.000000,0.000000,6,6,6.000000,0.000000,3.132505e+02,52.208416,38308.000000,0.000000,38308,38308,0,0.000000,0.000000,0,0,0,0.000000,0.000000,0,0,0,0,0,0,20,20,26.104208,26.104208,6,6,6.000000,0.000000,0.000000,0,0,0,0,1,1,0,0,1,9.000000,6.000000,6.000000,0,0,0,0,0,0,1,6,1,6,255,946,0,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
1,6,479,11,5,172,326,79,0,15.636364,31.449238,163,0,65.199997,89.278778,1.039666e+06,33402.922760,31.933332,25.510408,73,0,479,47.900002,38.942837,109,1,401,100.250000,101.736176,237,3,0,0,0,0,368,176,22964.509766,10438.413086,0,163,29.294117,56.529598,3195.595703,0,0,0,1,0,0,0,0,0,31.125000,15.636364,65.199997,0,0,0,0,0,0,11,172,5,326,29200,260,4,32,0.0,0.0,0,0,0.0,0.0,0,0,Benign
2,6,1095,10,6,3150,3150,1575,0,315.000000,632.561646,1575,0,525.000000,813.326477,5.753425e+06,14611.872150,73.000000,204.960968,810,1,1095,121.666664,298.746124,915,1,995,199.000000,345.535095,810,3,0,0,0,0,336,208,9132.419922,5479.452148,0,1575,370.588226,671.751526,451250.125000,0,0,0,1,0,0,0,0,0,393.750000,315.000000,525.000000,0,0,0,0,0,0,10,3150,6,3150,29200,2081,3,32,0.0,0.0,0,0,0.0,0.0,0,0,Benign
3,6,15206,17,12,3452,6660,1313,0,203.058823,425.778473,3069,0,555.000000,977.480347,6.650007e+05,1907.141918,543.071411,2519.931396,13391,0,15206,950.375000,3322.417725,13391,2,15112,1373.818237,4176.449707,13961,3,0,0,0,0,560,388,1117.979736,789.162170,0,3069,337.066681,704.654053,496537.375000,0,0,0,1,0,0,0,0,0,348.689667,203.058823,555.000000,0,0,0,0,0,0,17,3452,12,6660,29200,0,10,32,0.0,0.0,0,0,0.0,0.0,0,0,Benign
4,6,1092,9,6,3150,3152,1575,0,350.000000,694.509705,1576,0,525.333313,813.842896,5.771062e+06,13736.263740,78.000000,207.000931,794,1,1092,136.500000,313.850739,910,1,1015,203.000000,333.240143,794,3,0,0,0,0,304,208,8241.757812,5494.505371,0,1576,393.875000,704.585083,496440.125000,0,0,0,1,0,0,0,0,0,420.133331,350.000000,525.333313,0,0,0,0,0,0,9,3150,6,3152,29200,2081,2,32,0.0,0.0,0,0,0.0,0.0,0,0,Benign


### 6. Pregled dostupnih atributa (feature-a)

Dataset u pocetnom obliku sadrzi 78 kolona - 77 numerickih atributa i jednu kategoricku kolonu Label koja oznacava klasu kojoj instanca pripada (benigni saobracaj ili odredjeni tip napada). Numericki atributi opisuju karakteristike pojedinacnog mreznog toka (flow-a, definisanog kao niz paketa razmenjenih izmedju iste dve krajnje tacke tokom odredjenog vremenskog intervala) - primeri ukljucuju trajanje toka, ukupan broj i velicinu razmenjenih paketa, kao i brzinu prenosa podataka izrazenu u bajtovima ili paketima u sekundi.

In [ ]:
print(f"Broj kolona: {df.shape[1]}")
for i, col in enumerate(df.columns, 1):
    print(f"{i}. {col}")

Broj kolona: 78
1. Protocol
2. Flow Duration
3. Total Fwd Packets
4. Total Backward Packets
5. Fwd Packets Length Total
6. Bwd Packets Length Total
7. Fwd Packet Length Max
8. Fwd Packet Length Min
9. Fwd Packet Length Mean
10. Fwd Packet Length Std
11. Bwd Packet Length Max
12. Bwd Packet Length Min
13. Bwd Packet Length Mean
14. Bwd Packet Length Std
15. Flow Bytes/s
16. Flow Packets/s
17. Flow IAT Mean
18. Flow IAT Std
19. Flow IAT Max
20. Flow IAT Min
21. Fwd IAT Total
22. Fwd IAT Mean
23. Fwd IAT Std
24. Fwd IAT Max
25. Fwd IAT Min
26. Bwd IAT Total
27. Bwd IAT Mean
28. Bwd IAT Std
29. Bwd IAT Max
30. Bwd IAT Min
31. Fwd PSH Flags
32. Bwd PSH Flags
33. Fwd URG Flags
34. Bwd URG Flags
35. Fwd Header Length
36. Bwd Header Length
37. Fwd Packets/s
38. Bwd Packets/s
39. Packet Length Min
40. Packet Length Max
41. Packet Length Mean
42. Packet Length Std
43. Packet Length Variance
44. FIN Flag Count
45. SYN Flag Count
46. RST Flag Count
47. PSH Flag Count
48. ACK Flag Count
49. URG Fla

### 7. Analiza raspodele klasa

Sprovodi se provera broja instanci po svakoj klasi, cime se identifikuje eventualno prisustvo neizbalansiranosti podataka (situacije u kojoj pojedine klase imaju znacajno vise instanci od drugih) - problem koji, ukoliko se ne adresira odgovarajucim tehnikama, moze dovesti do toga da model tokom treniranja favorizuje dominantne klase na stetu retkih.

In [ ]:
label_col = [c for c in df.columns if 'label' in c.lower()][0]
print(f"Kolona sa labelom: '{label_col}'")
print()
print("Raspodela klasa:")
print(df[label_col].value_counts())

Kolona sa labelom: 'Label'

Raspodela klasa:
Label
Benign              391235
DoS Hulk            172846
DoS GoldenEye        10286
DoS slowloris         5385
DoS Slowhttptest      5228
Heartbleed              11
Name: count, dtype: int64


## Zakljucak beleznice

Ucitani su podaci iz fajla DoS-Wednesday-no-metadata.parquet, koji sadrze 584.991 instancu opisanu sa 78 atributa. Analizom raspodele klasa utvrdjeno je prisustvo pet relevantnih kategorija (Benign: 391.235 instanci; DoS Hulk: 172.846; DoS GoldenEye: 10.286; DoS slowloris: 5.385; DoS Slowhttptest: 5.228), kao i 11 instanci klase Heartbleed. Heartbleed predstavlja ranjivost u implementaciji OpenSSL biblioteke koja se eksploatise mehanizmom bitno razlicitim od DoS napada (curenje memorijskog sadrzaja putem manipulacije Heartbeat ekstenzijom TLS protokola, a ne preopterecenje resursa), zbog cega ne pripada predmetu ovog istrazivanja i bice iskljucena iz dataseta u narednom koraku.

Uocena je znacajna neizbalansiranost izmedju klasa, sa odnosom najbrojnije (Benign) i najredje klase (DoS Slowhttptest) od priblizno 75:1, sto zahteva primenu odgovarajucih mera prilikom pripreme podataka i treniranja modela (stratifikovana podela na skupove za treniranje i testiranje, a potencijalno i tehnike balansiranja klasa).

### 8. Cuvanje podataka

Radi izbegavanja ponovnog preuzimanja podataka sa Kaggle platforme u narednim beleznicama, ucitani dataset se cuva u parquet formatu u lokalnom `data` direktorijumu projekta. Parquet je kolonski orijentisan, komprimovan binarni format za skladistenje tabelarnih podataka koji omogucava efikasno cuvanje i citanje podataka.

**Sledeci korak:** Beleznica 02_preprocessing.ipynb, u okviru koje se sprovodi ciscenje podataka, enkodiranje kategorickih labela, podela na skupove za treniranje i testiranje, kao i skaliranje numerickih atributa.

In [ ]:
output_path = DATA_DIR / "raw_dos_wednesday.parquet"
df.to_parquet(output_path)

print("Sacuvano u:", output_path)

Mounted at /content/drive
Sacuvano na: /content/drive/MyDrive/Diplomski rad/data/raw_dos_wednesday.parquet
